[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/02_build_your_own_autograd.ipynb)

# ⚒️ Act I · Quest 02 — Build Your Own Autograd

*Forge a ~60-line engine that computes every gradient in one sweep — then train a real neural net with it.*

⬅️ [01_the_idea_of_learning](01_the_idea_of_learning.ipynb)  •  [03_tensors_the_real_metal](03_tensors_the_real_metal.ipynb) ➡️

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math, random

np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## The key insight

Any computation is a **graph** of tiny operations (`+`, `*`, `tanh`...), and each tiny op has a
derivative we know from high-school calculus. The **chain rule** lets us multiply those local
derivatives backwards through the graph to get the slope of the final loss with respect to
*every* input — in **one** backward pass, no matter how many knobs.

That's it. That's the secret inside `loss.backward()`. Let's build it.

### The plan
A `Value` wraps a number and remembers:
- `data` — the number,
- `grad` — the slope of the final output w.r.t. this value (filled in later),
- `_parents` and `_backward` — who made it, and *how to push gradient back to them*.

In [ ]:
class Value:
    """A scalar that remembers how it was made — and can compute gradients."""
    def __init__(self, data, _parents=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._parents = _parents
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad          # d(a+b)/da = 1
            other.grad += out.grad         # d(a+b)/db = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")
        def _backward():
            self.grad += (1 - t ** 2) * out.grad  # d tanh/dx = 1 - tanh^2
        out._backward = _backward
        return out

    def backward(self):
        # Topological sort: visit children before parents, then run the
        # chain rule backwards through the whole graph.
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for p in v._parents:
                    build(p)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    # conveniences so expressions read naturally
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __repr__(self): return f"Value({self.data:.4f}, grad={self.grad:.4f})"

### Watch it think

Let's trace `f = (a * b + c).tanh()` with `a=2, b=-3, c=10` and ask for all three gradients at once.

In [ ]:
a, b, c = Value(2.0), Value(-3.0), Value(10.0)
f = (a * b + c).tanh()
f.backward()

print("f     =", f.data)
print("df/da =", a.grad, "  (analytic: b * (1-tanh(ab+c)^2))")
print("df/db =", b.grad)
print("df/dc =", c.grad)

# Verify against finite differences — the slow method from Quest 01:
def numeric(name, delta=1e-5):
    vals = {"a": 2.0, "b": -3.0, "c": 10.0}
    up, dn = dict(vals), dict(vals)
    up[name] += delta; dn[name] -= delta
    g = lambda v: math.tanh(v["a"] * v["b"] + v["c"])
    return (g(up) - g(dn)) / (2 * delta)

for name, node in [("a", a), ("b", b), ("c", c)]:
    print(f"  {name}: engine={node.grad:.6f}  numeric={numeric(name):.6f}  ✓")

One `backward()` call, every gradient, and they match the finite-difference check. **You just
built the core of PyTorch.**

### Now the real test: train a neural network with YOUR engine

A neuron is `tanh(w·x + b)`. A layer is several neurons. An MLP is stacked layers. All built
from `+`, `*`, and `tanh` — exactly the ops your engine speaks.

In [ ]:
class Neuron:
    def __init__(self, n_in):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = self.b
        for wi, xi in zip(self.w, x):
            act = act + wi * xi
        return act.tanh()
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out):
        self.neurons = [Neuron(n_in) for _ in range(n_out)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, sizes):          # e.g. MLP([2, 4, 1])
        self.layers = [Layer(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

random.seed(1)
net = MLP([2, 4, 1])
print("your hand-forged network has", len(net.parameters()), "knobs")

### The XOR problem — impossible for a single neuron, easy for your MLP

XOR: output +1 when the two inputs differ, −1 when they match. It's the classic test that
killed single-layer networks in 1969 and was solved by backprop in 1986. Your engine *is* backprop.

In [ ]:
X = [[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]]
Y = [-1.0, 1.0, 1.0, -1.0]

losses = []
for step in range(300):
    # forward
    preds = [net(x) for x in X]
    loss = sum((p - y) * (p - y) for p, y in zip(preds, Y))

    # backward — YOUR engine at work
    for p in net.parameters():
        p.grad = 0.0            # remember Quest 01: fresh slopes each step
    loss.backward()

    # descend
    for p in net.parameters():
        p.data -= 0.1 * p.grad
    losses.append(loss.data)

print("final predictions:", [f"{net(x).data:+.2f}" for x in X], " targets:", Y)
plt.plot(losses); plt.title("Your engine, training a neural net on XOR")
plt.xlabel("step"); plt.ylabel("loss"); plt.show()

🎉 **You trained a neural network with an autograd engine you wrote from scratch.**

Everything PyTorch adds from here is *engineering*, not new ideas:
- tensors instead of scalars (do a million of these at once),
- GPU kernels (do them fast),
- a library of ops, layers, and optimizers (don't rewrite `Neuron` every day).

Next quest: meet the industrial version of what you just forged.

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("sub_works", 10,
    lambda v: isinstance(v, Value) and abs(v.data - 1.5) < 1e-9,
    "compute Value(4.0) - Value(2.5) — __sub__ is already defined via __add__ and __neg__")
_register("chain", 15,
    lambda g: abs(g - 12.0) < 1e-6,
    "y = x*x*x at x=2 -> dy/dx = 3x^2 = 12. Build it from Value(2.0), call .backward(), submit x.grad")
_register("xor_master", 20,
    lambda final_loss: final_loss < 0.05,
    "keep training (more steps, or lr=0.2) until the XOR loss is below 0.05, submit loss.data")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `sub_works` (10 XP) — `check("sub_works", Value(4.0) - Value(2.5))`.
- `chain` (15 XP) — build `y = x*x*x` with `x = Value(2.0)`, backprop, then `check("chain", x.grad)`.
- `xor_master` (20 XP) — push the XOR training loss under `0.05`, then `check("xor_master", loss.data)`.

In [ ]:
# ⚔️ your attempts here...

# xp_report()